# Sensitivity studies

<div style="
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 12px 16px;
    max-width: 500px;
    background-color: #f9f9f9;
">
  <strong>✈️⚡ Electric Aircraft Design</strong><br>
  <br>
  <strong>Author:</strong> Florent LUTZ<br>
  <strong>Affiliation:</strong> ISAE-SUPAERO / DCAS <br>
  <strong>Course:</strong> Electric Aircraft Design<br>
</div>

For this study, and as mentionned earlier, we will consider some aircraft from the Pipistrel manufacturer who share a common airframe. The reason for that choice is that, being certified aircraft, a wide range of performances and design data are openly accessible and can serve as validation data.

## Thermal variant: Pipistrel SW121

The Pipistrel SW121 is a two-seater, high-wing, single-engine aircraft made of composite material. It has fixed landing gear, and is primarily used for training. Although it can be fitted with different engines from the Rotax catalog, the version used for this study was the one equipped with a Rotax 912 engine. The powertrain architecture will be modeled as two tanks feeding fuel into an ICE. The Virus SW 121 has a composite airframe so a multiplicative factor of 0.85 for the mass of airframe components was considered [1]. The main inputs for the sizing process are given in the table below.


| Parameter                        | Value      |
|----------------------------------|------------| 
| Payload on the design mission    | 188 kg     |
| Design range                     | 513 nm     |
| Design mission cruise speed      | 119 KTAS   |
| Design mission cruise altitude   | 2000 ft    |
| Approach speed                   | 58.5 KTAS  |
| Wing aspect ratio                | 12.0       |
| Rotax 912 SFC                    | 256 g/kW/h |
| Propeller diameter               | 1.7 m      |

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed. It is used to silence some pointless FAST-OAD warnings
</div>

In [ ]:
%%capture
import fastoad.api as oad

oad.list_modules()

The next cell contains the code to run the sizing process for the design of the Pipistrel SW 121 as well as some post-processing. It doesn't need to be changed at this stage but can be if you want to use some of the other post-processing functions of FAST-OAD. The process will be run differently to make it easier to change some values if necessary.
<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed. It is used to silence some pointless FAST-OAD warnings
</div>

In [ ]:
# Import required packages
import pathlib
import warnings
import fastoad.api as oad
from fastga_he.gui.performances_viewer import PerformancesViewer

# Let's start by defining the path to some useful folders and files
DATA_FOLDER_PATH = "data"
WORK_FOLDER_PATH = "workdir"
RESULT_FOLDER_PATH = "results"

path_to_pipistrel_sw_121_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "pipistrel_sw_121_configuration.yml"
path_to_pipistrel_sw_121_input_file = pathlib.Path(DATA_FOLDER_PATH) / "pipistrel_sw_121_input.xml"
path_to_pipistrel_sw_121_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "pipistrel_sw_121_propulsion_data.csv"
path_to_pipistrel_sw_121_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "pipistrel_sw_121_mission_file.csv"

with warnings.catch_warnings():
    pipistrel_sw_121_process_configurator = oad.FASTOADProblemConfigurator(path_to_pipistrel_sw_121_configuration_file)
    pipistrel_sw_121_process = pipistrel_sw_121_process_configurator.get_problem()
    pipistrel_sw_121_process.write_needed_inputs(path_to_pipistrel_sw_121_input_file)
    pipistrel_sw_121_process.read_inputs()
    pipistrel_sw_121_process.setup()
    
    # ==== This line allows to change the design range of the aircraft, change it only if prompted ==== #
    pipistrel_sw_121_process.set_val("data:TLAR:range", units="nmi", val=513.8)
    # ================================================================================================= #

    pipistrel_sw_121_process.run_model()
    pipistrel_sw_121_process.write_outputs()

oad.variable_viewer(pipistrel_sw_121_process.output_file_path)
    
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_pipistrel_sw_121_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_pipistrel_sw_121_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Compare the outputs of the sizing process with the following reference values found in litterature:
<ul>
    <li>MTOW = 600 kg</li>
    <li>OWE = 349 kg</li>
    <li>Wing area = 9.51 m2</li>
    <li>Fuel burn on design mission = 63 kg</li>
</ul>    
</div>

## Electric variant: Pipistrel Velis Electro

The Pipistrel Velis Electro, although it shares similar geometric features with the Pipistrel SW 121 as shown in Figure 3 in the [introductory Notebook](00_Introduction.ipynb), is a full electric aircraft and, as such, has very different performances. The powertrain, as presented in the POH as well as the way it has been modeled for the purposes of this study, is illustrated in the figure below. In addition to the powertrain description, information on the electric motor and battery pack structure is available in the POH or on the manufacturer's website.

<p align="center">
  <img src="./resources/pipistrel_velis_electro_propulsive_architecture.png" width="900">
  <br>
  <em>Figure 7 : Propulsive system architecture for the Pipistrel Velis Electro, the components highlighted in red are the ones modeled in this study.</a></em>
</p>

The inputs we will use for the sizing process are presented in the Table below. The inputs for the Pipistrel SW 121 will also be reminded for comparison.

| Parameter                        | Pipistrel SW 121 | Pipistrel Velis Electro |
|----------------------------------|------------------| ------------------------|
| Payload on the design mission    | 188 kg           | 172 kg                  |
| Design range                     | 513 nm           | 28.6 nm                 |
| Design mission cruise speed      | 119 KTAS         | 93 KTAS                 |
| Design mission cruise altitude   | 2000 ft          | 2000 ft                 |
| Approach speed                   | 58.5 KTAS        | 58.5 KTAS               |
| Wing aspect ratio                | 12.0             | 12.0                    |
| Rotax 912 SFC                    | 256 g/kW/h       | NA                      |
| Propeller diameter               | 1.7 m            | 1.64 m                  |
| Battery structure                | NA               | 96 series               |

First, we will simply look at the reference case, analyze the outputs and compare it with the original design. Due to the high weight sensitivity of the electric design, as opposed to the previous case, a clever initialization of the design process will be required. 

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
Unless instructed otherwise, FAST-OAD-CS23-HE starts the design process with an initial guess of the MTOW at 1500 kg. In the case of an electric aircraft, this might lead to a divergence of the mass. To allow a change of the initial guess for the MTOW, small adjustments to the way the design process is launched is required.
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed at this stage. As discussed in the previous Notebook, this way of running the design process is equivalent to what we have done so far.
</div>

In [ ]:
path_to_pipistrel_electro_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "pipistrel_electro_configuration.yml"
path_to_pipistrel_electro_input_file = pathlib.Path(DATA_FOLDER_PATH) / "pipistrel_electro_input.xml"
path_to_pipistrel_electro_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "pipistrel_electro_propulsion_data.csv"
path_to_pipistrel_electro_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "pipistrel_electro_mission_file.csv"

# Generate the final input file for the hybrid design computation, ignoring all useless warnings
with warnings.catch_warnings():
    pipistrel_electro_process_configurator = oad.FASTOADProblemConfigurator(path_to_pipistrel_electro_configuration_file)
    pipistrel_electro_process = pipistrel_electro_process_configurator.get_problem()
    pipistrel_electro_process.write_needed_inputs(path_to_pipistrel_electro_input_file)
    pipistrel_electro_process.read_inputs()
    pipistrel_electro_process.setup()
    
    # Give good initial guess on a few key value the code will iterate on in order to reduce the time it takes to converge
    pipistrel_electro_process.set_val("data:weight:aircraft:MTOW", units="kg", val=600.0)
    pipistrel_electro_process.set_val("data:weight:aircraft:OWE", units="kg", val=400.0)
    pipistrel_electro_process.set_val("data:weight:aircraft:MZFW", units="kg", val=600.0)
    pipistrel_electro_process.set_val("data:weight:aircraft:ZFW", units="kg", val=600.0)
    pipistrel_electro_process.set_val("data:weight:aircraft:MLW", units="kg", val=600.0)
    
    # ==== This line allows to change the design range of the aircraft, change it only if prompted ==== #
    pipistrel_electro_process.set_val("data:TLAR:range", units="nmi", val=28.6)
    # ================================================================================================= #

    pipistrel_electro_process.run_model()
    pipistrel_electro_process.write_outputs()

oad.variable_viewer(pipistrel_electro_process.output_file_path)
    
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_pipistrel_electro_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_pipistrel_electro_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Compare the outputs of the sizing process with the following reference values found in litterature:
<ul>
    <li>MTOW = 600 kg</li>
    <li>OWE = 428 kg</li>
    <li>Wing area = 9.51 m2</li>
</ul>
    
Write down the value of the SOC at the end of the reserve (can be read in the variable viewer or performance viewer).<br>
Commment on the achievables ranges and cruise performances of the two designs.<br>
Assuming an energy density of 43.7 MJ/kg compare the energy required to perform each mission<br>
Now compute the energy required per nautical miles of flight and per hour of flight. Comment.
</div>

### Sensitivity to network voltage

The question of the network voltage is paramount as, for the same power, it allows for lower current, thus reducing Joule losses and cable cross sections. It will be briefly discussed and investigated here. Let's start by looking at the evolution of the voltage in the current configuration.

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
In the previous cell, display the evolution of the battery's open circuit voltage. (There are two battery packs but they are used the same way so whichever you select is fine). Comment about that evolution. Most importantly recall to what other battery metric it is linked.
</div>

To provide some clues to the next question, we will display in the next cell the powertrain modeled for that study. You can note the absence of power electronics to correct the voltage at the output of the batteries.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Import useful functions
from IPython.display import IFrame
from fastga_he.gui.power_train_network_viewer import power_train_network_viewer

# Define path to files of interest
path_to_pipistrel_electro_propulsive_architecture = pathlib.Path(DATA_FOLDER_PATH) / "pipistrel_electro_propulsive_architecture.yml"
path_to_pipistrel_electro_network_file = pathlib.Path(RESULT_FOLDER_PATH) / "pipistrel_electro_network_file.html"

power_train_network_viewer(path_to_pipistrel_electro_propulsive_architecture, path_to_pipistrel_electro_network_file)

IFrame(src=path_to_pipistrel_electro_network_file, width="100%", height="500px")

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Going back to the evolution of the state of the powertrain during the mission, display the evolution of the network voltage. You can do so by displaying <em>dc_splitter_1 dc_voltage</em>. Comment and explain the shape of the voltage knowing there is a direct connection between the battery packs and the network.
</div>

For the purpose of the next study, we will introduce a DC-DC converter at the output of both battery packs to mimic a correction of the voltage to a constant value troughout the mission. Initially, we will fix that value at 360V. The purpose of this case is to study the influence of the voltage, so to isolate other influences, we will assume very high power density and efficiencies for the DC-DC converters so they are "transparent" to the aircraft. We will name this derivative aircraft the Noctule Electro. It will be sized in the next cell.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
    The next cell should not be changed <em>right now</em>.
</div>

In [ ]:
path_to_noctule_electro_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "noctule_electro_configuration.yml"
path_to_noctule_electro_input_file = pathlib.Path(DATA_FOLDER_PATH) / "noctule_electro_input.xml"
path_to_noctule_electro_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "noctule_electro_propulsion_data.csv"
path_to_noctule_electro_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "noctule_electro_mission_file.csv"

# Generate the final input file for the hybrid design computation, ignoring all useless warnings
with warnings.catch_warnings():
    noctule_electro_process_configurator = oad.FASTOADProblemConfigurator(path_to_noctule_electro_configuration_file)
    noctule_electro_process = noctule_electro_process_configurator.get_problem()
    noctule_electro_process.write_needed_inputs(path_to_noctule_electro_input_file)
    noctule_electro_process.read_inputs()
    noctule_electro_process.setup()
    
    # Give good initial guess on a few key value the code will iterate on in order to reduce the time it takes to converge
    noctule_electro_process.set_val("data:weight:aircraft:MTOW", units="kg", val=600.0)
    noctule_electro_process.set_val("data:weight:aircraft:OWE", units="kg", val=400.0)
    noctule_electro_process.set_val("data:weight:aircraft:MZFW", units="kg", val=600.0)
    noctule_electro_process.set_val("data:weight:aircraft:ZFW", units="kg", val=600.0)
    noctule_electro_process.set_val("data:weight:aircraft:MLW", units="kg", val=600.0)
    
    # ========== These lines allow to change the network voltage, change it only if prompted ========== #
    noctule_electro_process.set_val("data:propulsion:he_power_train:DC_DC_converter:dc_dc_converter_1:voltage_out_target_mission", units="V", val=360)
    noctule_electro_process.set_val("data:propulsion:he_power_train:DC_DC_converter:dc_dc_converter_2:voltage_out_target_mission", units="V", val=360)
    # ================================================================================================= #
    
    # ==== This line allow to change the inverter switching frequencies, change it only if prompted === #
    noctule_electro_process.set_val("data:propulsion:he_power_train:inverter:inverter_1:switching_frequency_mission", units="Hz", val=12000)
    # ==== This line allow to change the inverter switching frequencies, change it only if prompted === #

    noctule_electro_process.run_model()
    noctule_electro_process.write_outputs()

oad.variable_viewer(noctule_electro_process.output_file_path)
    
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_noctule_electro_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_noctule_electro_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

Look at the network voltage and confirm that it is now constant. You will note that the value is slightly different than what we set as an input. This is due to the losses in the SSPC which translates into a voltage drop.

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Write down the value of the cable's conductor radius and the insulation thickness in mm. (Be careful, the values in the table above are rounded. You can double click on a value to see the raw data). In the performance viewer, look at the maximum value of losses in the SSPC. <br>
<strong>✏️ Action required</strong><br>
Now change the value of the network voltage by picking a few values between 360V and 800V and write down the values of the variables listed in the question above.
Comment on their evolution and the interest of increasing the voltage.
</div>

The next cell will show you the evolution of the cable's cross section.

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Complete the next cell with the values you wrote down.
</div>

In [ ]:
import numpy as np

network_voltages = np.array([..., ..., ...])  # [..., ..., ...]
conductor_radii = np.array( [..., ..., ...])  # In mm [..., ..., ...]
insulation_thicknesses = np.array([..., ..., ...])  # In mm [..., ..., ...]

# ====== The rest of the cell just serves to plot a representation of the cable ====== #
import plotly.graph_objects as go

total_radii = conductor_radii + insulation_thicknesses + 0.2 + 0.1  # For shielding tape and sheath
rounded_radius = np.max(np.ceil(total_radii))
center_x_positions = np.arange(0, 2.0 * len(network_voltages), 2) * rounded_radius
center_y_positions = np.full_like(network_voltages, rounded_radius)

fig = go.Figure()

for idx, voltage in enumerate(network_voltages):
    # Extract required values
    center_x_position = center_x_positions[idx]
    center_y_position = center_y_positions[idx]
    total_radius = total_radii[idx]
    conductor_radius = conductor_radii[idx]
    
    # First draw the outermost layer which is the mechanical sheath
    fig.add_shape(
        type="circle",
        xref="x", 
        yref="y",
        fillcolor="black",
        x0=center_x_position - total_radius, 
        y0=center_y_position - total_radius, 
        x1=center_x_position + total_radius, 
        y1=center_y_position + total_radius,
        line_color="black",
    )
    
    # Then, overlay the shielding tape
    fig.add_shape(
        type="circle",
        xref="x", 
        yref="y",
        fillcolor="rgba(184, 115, 51, 1)",
        x0=center_x_position - (total_radius - 0.1), 
        y0=center_y_position - (total_radius - 0.1), 
        x1=center_x_position + (total_radius - 0.1), 
        y1=center_y_position + (total_radius - 0.1),
        line_color="rgba(184, 115, 51, 1)",
    )
    # Then, overlay the insulation
    fig.add_shape(
        type="circle",
        xref="x", 
        yref="y",
        fillcolor="rgba(66, 133, 244, 1)",
        x0=center_x_position - (total_radius - 0.1 - 0.2), 
        y0=center_y_position - (total_radius - 0.1 - 0.2), 
        x1=center_x_position + (total_radius - 0.1 - 0.2), 
        y1=center_y_position + (total_radius - 0.1 - 0.2),
        line_color="rgba(66, 133, 244, 1)",
    )
    # Finally, add the copper core
    fig.add_shape(
        type="circle",
        xref="x", 
        yref="y",
        fillcolor="rgba(184, 115, 51, 1)",
        x0=center_x_position - conductor_radius, 
        y0=center_y_position - conductor_radius, 
        x1=center_x_position + conductor_radius, 
        y1=center_y_position + conductor_radius,
        line_color="rgba(184, 115, 51, 1)",
    )
    
    # Annotate
    fig.add_annotation(
        x=center_x_position,
        y=center_y_position - rounded_radius,
        text=str(voltage) + " V",
        showarrow=False,
        font=dict(size=20, color="black"),
        xanchor="center",
        yanchor="top",
    )

x_range = [np.min(center_x_positions) - rounded_radius, np.max(center_x_positions) + rounded_radius]
y_range = [-rounded_radius / 2, 2. * rounded_radius]
    
fig.update_xaxes(
    showline=False,
    showgrid=False,
    range=x_range,
    showticklabels=False
)
fig.update_yaxes(
    showline=False,
    showgrid=False,
    range=y_range,
    showticklabels=False
)
fig.update_layout(
    plot_bgcolor="white",
    margin=dict(l=5, r=5, t=60, b=5),
    height=400,
    width=int(350.0 * (x_range[-1] - x_range[0]) / (y_range[-1] - y_range[0])),
    title_font=dict(size=20),
    title_x=0.5,
    title_text="Cable cross section evolution",
)
fig.show()

### Sensitivity to switching frequency

Another interesting design driver is the switching frequency of the power controllers. Indeed, the higher the switching frequency of the semiconductor modules inside the power controller, the lower the current ripples from the conversion. Therefore, the filters can be made smaller, leading to lighter power controller. However, it comes at the cost of higher losses. As a reminder the switching losses in the controller are directly proportional to the number of time the switch opens and closes, thus on the switching frequency. Current technologies use switching frequencies of 12 kHz but future prospect have predicted values of around 40 kHz, and even 100 kHz [2].

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
<strong>✏️ Action required</strong><br>
Go back to the cell that runs the sizing process of the Noctule Electro and set the value of the network voltage back to 360 V. Now run the sizing process for three following values of the inverter switchig frequency: 12 kHz, 40 kHz, 100 kHz. Write down the mass of the inverter, the maximum losses of the inverter and the battery mass. Comment on the evolution
</div>

## Hydrogen variant: Noctule H2X

For the final aircraft we will investigate, we will size an alternative of the Pipistrel Velis Electro functionning on hydrogen. From here on, we will call this aircraft the Noctule H2X. Due to the expected low power demand and short range, and to ease the Entry into Service and operability, we will opt for gaseous hydrogen stored in a tank at 700 bar. The powertrain will be similar to that of the Pipistrel Velis Electro except for the batteries that will be replaced by a fuel cell located in the front and plugged into a tank at the back. This results in the powertrain shown [here](./data/noctule_h2x_propulsive_architecture.yml) and which can be visualized in the next cell. For more information about the models used for the hydrogen powertrain, you can check [FAST-OAD-CS23-HE documentation](https://fast-oad-cs23-he.readthedocs.io/en/latest/documentation/models/index.html#models-index).

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Define path to files of interest
path_to_noctule_h2x_propulsive_architecture = pathlib.Path(DATA_FOLDER_PATH) / "noctule_h2x_propulsive_architecture.yml"
path_to_noctule_h2x_network_file = pathlib.Path(RESULT_FOLDER_PATH) / "noctule_h2x_network_file.html"

power_train_network_viewer(path_to_noctule_h2x_propulsive_architecture, path_to_noctule_h2x_network_file)

IFrame(src=path_to_noctule_h2x_network_file, width="100%", height="500px")

The next cell will run the sizing process for the Noctule H2X. The current value of the range matches the value of the design range of the Pipistrel SW 121 and will need to be changed for an ulterior question. Leave it as is for the first execution. For the other TLARs, they are shown below. For the cruise speed, we will match that of the Velis Electro as it essentially a constraint on the power of the electric motor which we don't wan't to change. For the payload, since hydrogen should be lighter than batteries, we will take the payload of the SW 121. You are free to challenge this assumptions in regards to the results of future question.

| Parameter                        | Pipistrel SW 121 | Pipistrel Velis Electro | Noctule H2X |
|----------------------------------|------------------| ------------------------| ------------|
| Payload on the design mission    | 188 kg           | 172 kg                  | 188 kg      |
| Design range                     | 513 nm           | 28.6 nm                 | 513* nm     |
| Design mission cruise speed      | 119 KTAS         | 93 KTAS                 | 93 KTAS     |
| Design mission cruise altitude   | 2000 ft          | 2000 ft                 | 2000 ft     |
| Approach speed                   | 58.5 KTAS        | 58.5 KTAS               | 58.5 KTAS   |
| Wing aspect ratio                | 12.0             | 12.0                    | 12.0        |
| Rotax 912 SFC                    | 256 g/kW/h       | NA                      | NA          |
| Propeller diameter               | 1.7 m            | 1.64 m                  | 1.64        |
| Battery structure                | NA               | 96 series               | NA          |

In [ ]:
path_to_noctule_h2x_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "noctule_h2x_configuration.yml"
path_to_noctule_h2x_input_file = pathlib.Path(DATA_FOLDER_PATH) / "noctule_h2x_input.xml"
path_to_noctule_h2x_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "noctule_h2x_propulsion_data.csv"
path_to_noctule_h2x_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "noctule_h2x_mission_file.csv"

# Generate the final input file for the hybrid design computation, ignoring all useless warnings
with warnings.catch_warnings():
    noctule_h2x_process_configurator = oad.FASTOADProblemConfigurator(path_to_noctule_h2x_configuration_file)
    noctule_h2x_process = noctule_h2x_process_configurator.get_problem()
    noctule_h2x_process.write_needed_inputs(path_to_noctule_h2x_input_file)
    noctule_h2x_process.read_inputs()
    noctule_h2x_process.setup()
    
    # Give good initial guess on a few key value the code will iterate on in order to reduce the time it takes to converge
    noctule_h2x_process.set_val("data:weight:aircraft:MTOW", units="kg", val=600.0)
    noctule_h2x_process.set_val("data:weight:aircraft:OWE", units="kg", val=400.0)
    noctule_h2x_process.set_val("data:weight:aircraft:MZFW", units="kg", val=600.0)
    noctule_h2x_process.set_val("data:weight:aircraft:ZFW", units="kg", val=600.0)
    noctule_h2x_process.set_val("data:weight:aircraft:MLW", units="kg", val=600.0)
    
    # ==== This line allows to change the design range of the aircraft, change it only if prompted ==== #
    noctule_h2x_process.set_val("data:TLAR:range", units="nmi", val=513.0) 
    # ================================================================================================= #
    
    noctule_h2x_process.run_model()
    noctule_h2x_process.write_outputs()

oad.variable_viewer(noctule_h2x_process.output_file_path)
    
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_noctule_h2x_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_noctule_h2x_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
<strong>✏️ Action required</strong><br>
Go back to the cell that runs the sizing process of the Noctule H2X (the previous cell) and find the value of the design range that gives roughly the same MTOW as that of the two other aircraft in the family. Comment on the weight of the fuel cell stack, the hydrogen tank mass and the hydrogen mass. Compute the gravimetric index and compare with state of the art values.
</div>

### Preliminary considerations about the thermal management system (TMS)

Another major issue with the use of electricity on-board an aircraft is the need to evacuate all the heat the components produce, which, for a conventional fuel-based architecture is generally done through the exhaust of hot gases. At the MW order of magnitude, even with the high efficiency of electric components, the amount of waste heat becomes significant. At this magnitude and even assuming near-future efficiencies of 97\%, the heat induced by losses is equivalent to the heat produced by a barbecue grill [3]. This implies that components that do not directly contribute to the generation of thrust will need to be carried on the aircraft for cooling. This not only adds weight (up to 17\% of the propulsion system weight depending on the configuration) but also adds drag and power consumption. Thermal management systems consists of a complex architecture of coolant pipes, heat exchangers, pumps and air inlets. An example is given in the Figure below.

<p align="center">
  <img src="./resources/example_tms_architecture.PNG" width="900">
  <br>
  <em>Figure 8 : Example of TMS architecture [4] </a></em>
</p>


The final configuration we will consider is a derivative of the Noctule H2X which we will call the Noctule H2X2 and which will include some preliminary considerations regarding the sizing of the thermal management system. Namely it will include an additional electric load which will represent the thermal management system. This load will add an additional power consumption to the aircraft and add weight as well as drag, therefore reducing the achievable range. All in all, the difference with respect to the Noctule H2X will be:
- The fuel cell rated power will be increased by the value of the TMS power consumption
- The fuel cell will need to provide, all along the mission, additional power equivalent to the TMS power consumption
- The propulsive system will be heavier due to the TMS
- The aircraft drag will be increased.

To account for those effects we will make the following considerations inspired by the results from [4]. It is worth noting that the results presented in [4] are given for a specific architecture using liquid hydrogen where, in addition to cooling the fuel cell, the hydrogen must be reheated. The contribution to the power drawn of that last process, however, is negligible when compared to the heat required to cool the fuel cell. Regarding the TMS weight, we will consider it to be a bit lighter that what was shown in [4]. The following assumptions are taken:
- The TMS will have a constant power consumption equal to 0.14 kW per kW of fuel cell rated power
- The TMS's weight will be computed as 0.4 kg per kW of fuel cell rated power
- The aircraft cooling drag coefficient will be increased by 1.58x10**-5 by kW of fuel cell rated

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Start by computing the new fuel cell rated power considering the rated power of the Noctule H2X fuel cell (which you can find using the variable viewer) and the power required for the TMS. We will consider that the propulsive power remains constant.<br> 
Then compute the power consumption of the TMS.<br>
From the new rated power of the fuel cell compute the weight of the TMS.<br>
From the power consumption of the TMS (assumed constant) and the weight of the TMS, compute the "power density" of the TMS (kW of power consumed by the TMS/kg of TMS).<br>
Finally compute the corresponding increase in aircraft cooling drag.<br>
</div>

The next cell will the sizing process for the Noctule H2X2.

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Modify the parameters left as ... in the <strong>next cell</strong> before executing it with the values you have computed.
</div>

In [ ]:
path_to_noctule_h2x2_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "noctule_h2x2_configuration.yml"
path_to_noctule_h2x2_input_file = pathlib.Path(DATA_FOLDER_PATH) / "noctule_h2x2_input.xml"
path_to_noctule_h2x2_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "noctule_h2x2_propulsion_data.csv"
path_to_noctule_h2x2_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "noctule_h2x2_mission_file.csv"

# Generate the final input file for the hybrid design computation, ignoring all useless warnings
with warnings.catch_warnings():
    noctule_h2x2_process_configurator = oad.FASTOADProblemConfigurator(path_to_noctule_h2x2_configuration_file)
    noctule_h2x2_process = noctule_h2x2_process_configurator.get_problem()
    noctule_h2x2_process.write_needed_inputs(path_to_noctule_h2x2_input_file)
    noctule_h2x2_process.read_inputs()
    noctule_h2x2_process.setup()
    
    # Give good initial guess on a few key value the code will iterate on in order to reduce the time it takes to converge
    noctule_h2x2_process.set_val("data:weight:aircraft:MTOW", units="kg", val=600.0)
    noctule_h2x2_process.set_val("data:weight:aircraft:OWE", units="kg", val=400.0)
    noctule_h2x2_process.set_val("data:weight:aircraft:MZFW", units="kg", val=600.0)
    noctule_h2x2_process.set_val("data:weight:aircraft:ZFW", units="kg", val=600.0)
    noctule_h2x2_process.set_val("data:weight:aircraft:MLW", units="kg", val=600.0)
    
    # ==== This line allows to change the design range of the aircraft, change it only if prompted ==== #
    noctule_h2x2_process.set_val("data:TLAR:range", units="nmi", val=513.0)
    # ================================================================================================= #
    
    # ============== This line allows to account for the TMS, change it only if prompted ============== #
    # For the nexy line, the term "power density" might be a stretch. It is actually kW of power consumed by the TMS/kg of TMS  
    noctule_h2x2_process.set_val("data:propulsion:he_power_train:aux_load:dc_aux_load_1:power_density", units="kW/kg", val=...)
    noctule_h2x2_process.set_val("data:propulsion:he_power_train:aux_load:dc_aux_load_1:power_in_mission", units="kW", val=...)
    
    noctule_h2x2_process.set_val("data:propulsion:he_power_train:PEMFC_stack:pemfc_stack_1:power_rating", units="kW", val=...)
    
    noctule_h2x2_process.set_val("data:aerodynamics:cooling:cruise:CD0", val=0.0005525 + ...)
    noctule_h2x2_process.set_val("data:aerodynamics:cooling:cruise:CD0", val=0.0005525 + ...)
    # ================================================================================================= #
    
    noctule_h2x2_process.run_model()
    noctule_h2x2_process.write_outputs()

oad.variable_viewer(noctule_h2x2_process.output_file_path)
    
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_noctule_h2x2_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_noctule_h2x2_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Write down the value of the Noctule H2X2's MTOW. Compare it to the Noctule H2X. Compare the hydrogen consumption and the weight of the TMS.<br>
<strong>✏️ Action required</strong><br>
Now go back to the cell that runs the sizing process of the Noctule H2X2 and find the value of the range that give a value of the MTOW similar to that of the original aircraft. Compare it to the range of the Noctule H2X. Conclude about the importance of the TMS.
</div>

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
There are few things to note regarding some of the assumptions we have made in this work. The first of which is that we have assumed that the additionalt power consumption due to the TMS was constant. In practice, it varies according to several other mission parameters like the efficiencies, the power drawn and the outside air temperature and density. It is also worth noting that the numbers we used stemmed from a case widely different than the Noctule H2X2. The reference case in [4] is a Kodiak 900, with higher power demands, flies at higher altitude and most importantly, the reference case considers a liquid hydrogen powertrain.
</div>

## Bibliography

[1] Raymer, Daniel. Aircraft design: a conceptual approach. American Institute of Aeronautics and Astronautics, Inc., 2012.

[2] Pohl, Markus, et al. "Preliminary design of integrated partial turboelectric aircraft propulsion systems." Journal of the Global Power and Propulsion Society 6 (2022): 1-23.

[3] Freeman, Jeffrey, et al. "Challenges and opportunities for electric aircraft thermal management." Aircraft Engineering and Aerospace Technology: An International Journal 86.6 (2014): 519-524.

[4] Habrard, Valentine. Conception préliminaire, intégration et évaluation d'un système de gestion thermique à base de liquide pour les avions hybrides à pile à combustible. Diss. Toulouse, ISAE, 2025.